# Simulation Confidence Analysis

This notebook demonstrates the confidence analysis tools for quantifying
uncertainty in Monte Carlo simulation results. The key question: **how many
simulations do we need to be confident in the 95th percentile?**

Two workflows are available:

1. **Planning** (before simulation) — uses exact portfolio moments to
   deterministically recommend a simulation count. No simulation needed.
2. **Validation** (after simulation) — uses order statistics from simulation
   output to confirm the precision achieved.

The analytical approach accounts for heterogeneous mortality rates (individual
qx values) and volume concentration via the Cornish-Fisher expansion.

See [docs/confidence_analysis.md](../docs/confidence_analysis.md) for the
full statistical methodology.

In [1]:
import numpy as np
import pandas as pd

from mortality_simulations import (
    stochastic_runs_hybrid,
    plan_simulation_count,
    compute_portfolio_moments,
    estimate_quantile_ci_width,
    analyze_simulation_confidence,
    estimate_required_simulations,
    generate_confidence_summary,
    print_confidence_summary,
)

## 1. Create a sample portfolio

In [2]:
np.random.seed(42)
n_lives = 1_000

data = pd.DataFrame({
    "volume":      np.random.uniform(100_000, 50_000_000, n_lives),
    "baseline_qx": np.random.uniform(0.001, 0.05, n_lives),
    "shocked_qx":  np.random.uniform(0.002, 0.08, n_lives),
})

print(f"Portfolio: {n_lives:,} lives")
print(f"Volume range: {data['volume'].min():,.0f} - {data['volume'].max():,.0f}")
print(f"Shocked qx range: {data['shocked_qx'].min():.4f} - {data['shocked_qx'].max():.4f}")
data.describe()

Portfolio: 1,000 lives
Volume range: 331,138 - 49,985,912
Shocked qx range: 0.0020 - 0.0798


,volume,baseline_qx,shocked_qx
count,1.000000e+03,1000.000000,1000.000000
mean,2.456380e+07,0.025844,0.041188
std,1.457765e+07,0.014317,0.022673
min,3.311379e+05,0.001158,0.002001
25%,1.187507e+07,0.012813,0.022385
50%,2.489069e+07,0.026418,0.041048
75%,3.724155e+07,0.038263,0.061210
max,4.998591e+07,0.049971,0.079830


## 2. Planning: How many simulations do I need?

**Start here.** `plan_simulation_count` is the recommended entry point.
It computes exact portfolio moments and projects CI widths for different
simulation counts — all deterministically, with no simulation required.

This accounts for heterogeneous mortality rates and volume concentration
using the Cornish-Fisher expansion for skewness correction.

In [3]:
plan = plan_simulation_count(
    data["volume"].values,
    data["shocked_qx"].values,
    target_ci_width_relative=1.0,  # Target: 1% CI width for the 95th pctile
)

print(f"Recommended: {plan['recommended_n']:,} simulations for ≤1% CI width\n")

target = plan["target_ci_width_relative"]
print(f"{'Simulations':>15}  {'CI Width (%)':>12}  {'Meets Target':>12}")
print("-" * 45)
for s in plan["scenarios"]:
    meets = s["ci_width_relative"] <= target
    marker = "  <--" if s["n_simulations"] == plan["recommended_n"] else ""
    print(f"{s['n_simulations']:>15,}  {s['ci_width_relative']:>11.3f}%  {'yes' if meets else 'no':>12}{marker}")

m = plan["portfolio_moments"]
print(f"\nPortfolio moments:")
print(f"  Mean:     {m['mean']:>20,.0f}")
print(f"  Std Dev:  {m['std']:>20,.0f}")
print(f"  CV:       {m['cv']:>20.4f}")
print(f"  Skewness: {m['skewness']:>20.4f}")

Recommended: 100,000 simulations for ≤1% CI width

    Simulations  CI Width (%)  Meets Target
---------------------------------------------
          1,000        3.915%            no
         10,000        1.238%            no
        100,000        0.392%           yes  <--
      1,000,000        0.124%           yes

Portfolio moments:
  Mean:            1,016,518,699
  Std Dev:           178,977,068
  CV:                     0.1761
  Skewness:               0.1874


## 3. Under the hood: portfolio moments and CI width projection

`plan_simulation_count` calls two lower-level functions that you can use
directly for more control:

- `compute_portfolio_moments()` — exact mean, variance, skewness from
  the weighted Poisson binomial model
- `estimate_quantile_ci_width()` — CI width at any simulation count,
  using the Cornish-Fisher corrected density

In [4]:
moments = compute_portfolio_moments(
    data["volume"].values,
    data["shocked_qx"].values,
)

print("Exact portfolio moments:")
for k, v in moments.items():
    if isinstance(v, float):
        print(f"  {k:>25}: {v:>24,.4f}")
    else:
        print(f"  {k:>25}: {v}")

print()
print(f"{'Simulations':>15}  {'CI Width (abs)':>20}  {'CI Width (%)':>12}  {'SE':>18}")
print("-" * 72)

for n in [1_000, 10_000, 100_000, 1_000_000]:
    ci = estimate_quantile_ci_width(moments, n_simulations=n)
    print(
        f"{n:>15,}  "
        f"{ci['ci_width_absolute']:>20,.0f}  "
        f"{ci['ci_width_relative']:>11.3f}%  "
        f"{ci['se_quantile']:>18,.0f}"
    )

print()
ci = estimate_quantile_ci_width(moments, n_simulations=10_000)
print(f"Quantile estimate (normal):         {ci['quantile_estimate_normal']:>20,.0f}")
print(f"Quantile estimate (Cornish-Fisher):  {ci['quantile_estimate_cf']:>20,.0f}")

Exact portfolio moments:
                       mean:       1,016,518,698.9056
                   variance: 32,032,791,041,666,608.0000
                        std:         178,977,068.4799
                   skewness:                   0.1874
       third_central_moment: 1,074,631,304,439,427,256,811,520.0000
                    n_lives: 1000
                         cv:                   0.1761

    Simulations        CI Width (abs)  CI Width (%)                  SE
------------------------------------------------------------------------
          1,000            51,701,019        3.915%          13,189,278
         10,000            16,349,298        1.238%           4,170,816
        100,000             5,170,102        0.392%           1,318,928
      1,000,000             1,634,930        0.124%             417,082

Quantile estimate (normal):                1,310,909,779
Quantile estimate (Cornish-Fisher):         1,320,445,999


## 4. Validation: run simulation and check empirical CI

Now run simulations (using the recommended count from step 2) and verify
precision with `analyze_simulation_confidence`, which uses distribution-free
order statistics for the CI.

In [5]:
n_sims = min(plan["recommended_n"], 10_000)  # Cap at 10K for this demo
print(f"Running {n_sims:,} simulations...\n")

results = stochastic_runs_hybrid(
    data,
    n_trials=n_sims,
    volume_col="volume",
    baseline_qx_col="baseline_qx",
    shocked_qx_col="shocked_qx",
)

ci = analyze_simulation_confidence(results, "claim_volume_shocked")
print(f"Empirical 95th percentile: {ci['point_estimate']:,.0f}")
print(f"95% CI: [{ci['ci_lower']:,.0f}, {ci['ci_upper']:,.0f}]")
print(f"CI width (relative): {ci['ci_width_relative']:.2f}%")

Running 10,000 simulations...



Empirical 95th percentile: 1,322,717,357
95% CI: [1,315,777,984, 1,329,936,564]
CI width (relative): 1.07%


## 5. Full summary: empirical vs analytical side-by-side

Passing `portfolio_data` to `generate_confidence_summary` enables the
analytical path alongside the empirical projections — giving a complete
picture of achieved and projected precision.

In [6]:
summary = generate_confidence_summary(
    results,
    "claim_volume_shocked",
    portfolio_data={
        "volumes": data["volume"].values,
        "qx": data["shocked_qx"].values,
    },
)

print_confidence_summary(summary)


SIMULATION CONFIDENCE ANALYSIS
Metric: claim_volume_shocked
Quantile: 95th percentile
Confidence Level: 95%

PORTFOLIO STRUCTURE (1,000 lives)
--------------------------------------------------
  Analytical mean:        1,016,518,698.91
  Analytical std dev:       178,977,068.48
  Coeff. of variation:             0.1761
  Skewness:                         0.1874

  Quantile estimates:
    Simulation (empirical):  1,322,717,357.40
    Normal approximation:    1,310,909,779.14
    Cornish-Fisher adjusted: 1,320,445,998.78

CURRENT RESULTS (10,000 simulations)
--------------------------------------------------
  Point estimate:         1,322,717,357.40
  CI lower bound:         1,315,777,983.80
  CI upper bound:         1,329,936,563.64
  CI width:                  14,158,579.84
  CI width (relative):               1.07%

PROJECTED CI WIDTH BY SIMULATION COUNT
----------------------------------------------------------------------
      Simulations    Empirical (%)   Analytical (%)  Statu

## 6. Pilot-based extrapolation (alternative to analytical planning)

`estimate_required_simulations` extrapolates from a pilot run's empirical
CI width. This is useful when you already have simulation results but didn't
use the planning workflow.

**Caveat**: pilot estimates are noisy for small pilot sizes. The function
warns when the pilot has fewer than 1,000 simulations and reports a
`pilot_stability` classification. For deterministic planning,
`plan_simulation_count` (section 2) is preferred.

In [7]:
import warnings

# Use the existing 10K simulation results as a "pilot"
print(f"Using {ci['n_simulations']:,} simulation results as pilot\n")

for target in [2.0, 1.0, 0.5]:
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        est = estimate_required_simulations(
            results, "claim_volume_shocked",
            target_ci_width_relative=target,
        )
        print(
            f"  Target {target}% CI width -> {est['estimated_n_required']:>10,} sims "
            f"({est['scaling_factor']:.1f}x)  "
            f"[stability: {est['pilot_stability']}]"
        )

# Show what happens with a very small pilot
print("\nSmall pilot (500 sims) — expect a warning:")
pilot_small = stochastic_runs_hybrid(
    data, n_trials=500,
    volume_col="volume", baseline_qx_col="baseline_qx",
    shocked_qx_col="shocked_qx",
)
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    est = estimate_required_simulations(
        pilot_small, "claim_volume_shocked",
        target_ci_width_relative=1.0,
    )
    if w:
        print(f"  Warning: {w[0].message}")
    print(f"  Stability: {est['pilot_stability']}")
    print(f"  Estimated n: {est['estimated_n_required']:,}")

Using 10,000 simulation results as pilot



  Target 2.0% CI width ->      2,865 sims (0.3x)  [stability: high]
  Target 1.0% CI width ->     11,458 sims (1.1x)  [stability: high]
  Target 0.5% CI width ->     45,832 sims (4.6x)  [stability: high]

Small pilot (500 sims) — expect a warning:
  Stability: low
  Estimated n: 17,928


## 7. Volume concentration impact

A portfolio with a few very large policies ("whales") has higher CV and
skewness, requiring substantially more simulations for the same precision.
The analytical approach captures this directly through the $v_i^2$ and
$v_i^3$ terms in the variance and skewness formulae.

In [8]:
np.random.seed(42)
n = 1_000

# Portfolio A: uniform volumes and qx
vol_uniform = np.ones(n) * 5_000_000
qx_uniform = np.ones(n) * 0.02

# Portfolio B: concentrated volumes (10 whale policies)
vol_concentrated = np.random.exponential(2_000_000, n)
vol_concentrated[:10] = np.random.uniform(100_000_000, 500_000_000, 10)
qx_varied = np.random.uniform(0.001, 0.06, n)

print(f"{'':>30} {'Uniform':>18} {'Concentrated':>18}")
print("-" * 68)

moments_a = compute_portfolio_moments(vol_uniform, qx_uniform)
moments_b = compute_portfolio_moments(vol_concentrated, qx_varied)

print(f"{'Mean':>30} {moments_a['mean']:>18,.0f} {moments_b['mean']:>18,.0f}")
print(f"{'Std Dev':>30} {moments_a['std']:>18,.0f} {moments_b['std']:>18,.0f}")
print(f"{'CV':>30} {moments_a['cv']:>18.4f} {moments_b['cv']:>18.4f}")
print(f"{'Skewness':>30} {moments_a['skewness']:>18.4f} {moments_b['skewness']:>18.4f}")

# Compare planning recommendations
plan_a = plan_simulation_count(vol_uniform, qx_uniform, target_ci_width_relative=1.0)
plan_b = plan_simulation_count(vol_concentrated, qx_varied, target_ci_width_relative=1.0)

print()
print(f"Recommended sims for ≤1% CI:")
print(f"  Uniform:      {plan_a['recommended_n']:>10,}")
print(f"  Concentrated: {plan_b['recommended_n']:>10,}")

print()
print("95th percentile CI width:")
print(f"{'Simulations':>15}  {'Uniform CI (%)':>15}  {'Concentrated CI (%)':>20}")
print("-" * 55)
for n_sim in [10_000, 100_000, 1_000_000]:
    ci_a = estimate_quantile_ci_width(moments_a, n_sim)
    ci_b = estimate_quantile_ci_width(moments_b, n_sim)
    print(
        f"{n_sim:>15,}  "
        f"{ci_a['ci_width_relative']:>14.3f}%  "
        f"{ci_b['ci_width_relative']:>19.3f}%"
    )

                                          Uniform       Concentrated
--------------------------------------------------------------------
                          Mean        100,000,000        178,480,615
                       Std Dev         22,135,944        205,400,314
                            CV             0.2214             1.1508
                      Skewness             0.2168             1.7167

Recommended sims for ≤1% CI:
  Uniform:         100,000
  Concentrated:  1,000,000

95th percentile CI width:
    Simulations   Uniform CI (%)   Concentrated CI (%)
-------------------------------------------------------
         10,000           1.489%                5.357%
        100,000           0.471%                1.694%
      1,000,000           0.149%                0.536%


## 8. Baseline vs shocked comparison

Compare confidence for different metrics from the same simulation run.

In [9]:
metrics = [
    ("claim_volume_baseline", "baseline_qx"),
    ("claim_volume_shocked", "shocked_qx"),
]

for metric, qx_col in metrics:
    ci_emp = analyze_simulation_confidence(results, metric)
    m = compute_portfolio_moments(data["volume"].values, data[qx_col].values)
    ci_ana = estimate_quantile_ci_width(m, n_simulations=10_000)

    print(f"{metric}")
    print(f"  Empirical 95th pctile: {ci_emp['point_estimate']:>20,.0f}")
    print(f"  Analytical (CF):       {ci_ana['quantile_estimate_cf']:>20,.0f}")
    print(f"  Empirical CI width:    {ci_emp['ci_width_relative']:>19.2f}%")
    print(f"  Analytical CI width:   {ci_ana['ci_width_relative']:>19.2f}%")
    print(f"  Portfolio skewness:    {m['skewness']:>19.4f}")
    print()

claim_volume_baseline
  Empirical 95th pctile:          887,643,375
  Analytical (CF):                887,529,758
  Empirical CI width:                   1.64%
  Analytical CI width:                  1.52%
  Portfolio skewness:                 0.2445

claim_volume_shocked
  Empirical 95th pctile:        1,322,717,357
  Analytical (CF):              1,320,445,999
  Empirical CI width:                   1.07%
  Analytical CI width:                  1.24%
  Portfolio skewness:                 0.1874

